In [4]:
import sys
sys.path.append("..") 

import os
import pandas as pd
import numpy as np
from scripts.embedding_pipeline_st import EmbeddingPipeline
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
input_dir = "../data/"
output_dir = os.path.expanduser("~/data/outputs/")
os.makedirs(output_dir, exist_ok=True)

In [7]:
df = pd.read_parquet(input_dir + 'amazon_products.parquet')
print("Data loaded successfully!")

Data loaded successfully!


In [8]:
pipeline = EmbeddingPipeline(model_name='all-mpnet-base-v2')

# Run and Save
embeddings = pipeline.compute_embeddings(
    df=df, 
    column='title', 
    batch_size=128, 
    save_path=save_path
)

Loading model 'all-mpnet-base-v2' on device: CUDA


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6859.13it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Preparing data for column: title
Starting embedding computation for 548552 items...


Batches: 100%|██████████| 4286/4286 [03:14<00:00, 22.00it/s]


Computation complete. Embedding shape: (548552, 768)
Embeddings saved to: outputs/title_embeddings_mpnet.npy


In [10]:
def test_embeddings(df, embeddings, query_index, top_k=5):
    # Get the title and embedding for the "query" product
    query_title = df.iloc[query_index]['title']
    query_vec = embeddings[query_index].reshape(1, -1)
    
    print(f"--- QUERY PRODUCT (Index {query_index}) ---")
    print(f"Title: {query_title}\n")
    
    scores = cosine_similarity(query_vec, embeddings).flatten()
    indices = np.argsort(scores)[-(top_k + 1):][::-1]
    
    print(f"--- TOP {top_k} SIMILAR PRODUCTS ---")
    for i in indices:
        if i == query_index:
            continue
            
        similarity = scores[i]
        match_title = df.iloc[i]['title']
        print(f"[{similarity:.4f}] {match_title}")

In [13]:
test_index = 100 
test_embeddings(df, embeddings, query_index=test_index)

--- QUERY PRODUCT (Index 100) ---
Title: Guide to Effective Staff Development in Health Care Organizations : A Systems Approach to Successful Training (J-B AHA Press)

--- TOP 5 SIMILAR PRODUCTS ---
[0.7756] A New Vision for Staff Development
[0.7008] Creating the Conditions for Teaching and Learning: A Handbook of Staff Development Activities
[0.6987] Redefining Staff Development : A Collaborative Model for Teachers and Administrators
[0.6982] Employee Training & Development
[0.6952] Human Resources in Healthcare: Managing for Success
